In [3]:
import dspy
print(dspy.__version__)

3.0.0


In [ ]:
import retrieve_dspy

query_writer = retrieve_dspy.MultiQueryWriter(
    collection_name="FreshstackLangchain",
    target_property_name="docs_text",
    retrieved_k=10,
    verbose=False
)

query_writer("How can I use Weaviate with LangChain?")

Prediction(
    final_answer='',
    sources=[Source(object_id='9608f261-9b02-4a9e-ab23-54b3b4c704f8'), Source(object_id='4cf9a2cb-e9ec-4883-8075-05054f38f8a6'), Source(object_id='acdb9703-2471-4b96-989b-b1fe0f1b8aff'), Source(object_id='250479d5-7312-4a56-8b97-edfa2b8e54b4'), Source(object_id='9043a9eb-adcc-4712-a459-9b9b2280c862'), Source(object_id='eabdeb84-50fd-43c0-8711-a0ec0a3d34b2'), Source(object_id='b980876d-0521-4440-abf5-ea2b64dc96ff'), Source(object_id='5163bd72-2249-4fa0-9ac4-7ba904a7f4e4'), Source(object_id='653f7f19-da48-45df-b9d5-19ef173390dc'), Source(object_id='6e8fc16b-f6b0-43b4-ad85-4e4ab00c9881'), Source(object_id='acdb9703-2471-4b96-989b-b1fe0f1b8aff'), Source(object_id='9608f261-9b02-4a9e-ab23-54b3b4c704f8'), Source(object_id='4cf9a2cb-e9ec-4883-8075-05054f38f8a6'), Source(object_id='2c9c4348-53cf-4f35-b070-b6de187aaa5b'), Source(object_id='250479d5-7312-4a56-8b97-edfa2b8e54b4'), Source(object_id='653f7f19-da48-45df-b9d5-19ef173390dc'), Source(object_id='1d04c3e1

In [ ]:
import os

import weaviate

from retrieve_dspy.metrics import create_coverage_metric_with_feedback
from retrieve_dspy.datasets.in_memory import load_queries_in_memory

trainset, testset = load_queries_in_memory(
    dataset_name="freshstack-langchain",
    train_samples=30,
    test_samples=20
)

weaviate_client = weaviate.connect_to_weaviate_cloud(
    cluster_url=os.getenv("WEAVIATE_URL"),
    auth_credentials=weaviate.auth.AuthApiKey(os.getenv("WEAVIATE_API_KEY")),
)

metric_for_gepa = create_coverage_metric_with_feedback(
    weaviate_client=weaviate_client,
    dataset_name="freshstack-langchain"
)

evaluator = retrieve_dspy.utils.get_evaluator(
    testset=testset,
    metric=metric_for_gepa
)

retrieve_dspy.utils.save_training_questions(trainset, "gepa_multi_query_writer_training_samples.jsonl")

/Users/cshorten/Desktop/retrieve-dspy/.venv/lib/python3.11/site-packages/weaviate/warnings.py:292: ResourceWarning: Con004: The connection to Weaviate was not closed properly. This can lead to memory leaks.
            Please make sure to close the connection using `client.close()`.
  warnings.warn(
/var/folders/41/8dp_379x15d8zz4ppsjthdw40000gn/T/ipykernel_71135/975551085.py:24: ResourceWarning: unclosed <ssl.SSLSocket fd=140, family=2, type=1, proto=0, laddr=('10.0.0.233', 49765), raddr=('35.201.124.182', 443)>
  evaluator = retrieve_dspy.utils.get_evaluator(


{'path': 'gepa_multi_query_writer_training_samples.jsonl',
 'added': 30,
 'total_in_file': 30}

In [13]:
trainset[0]

Example({'question': 'How should I add a field to the metadata of Langchain\'s Documents?\nFor example, using the CharacterTextSplitter gives a list of Documents:\nconst splitter = new CharacterTextSplitter({\n  separator: " ",\n  chunkSize: 7,\n  chunkOverlap: 3,\n});\nsplitter.createDocuments([text]);\n\nA document will have the following structure:\n{\n  "pageContent": "blablabla",\n  "metadata": {\n    "name": "my-file.pdf",\n    "type": "application/pdf",\n    "size": 12012,\n    "lastModified": 1688375715518,\n    "loc": { "lines": { "from": 1, "to": 3 } }\n  }\n}\n\nAnd I want to add a field to the metadata\n', 'dataset_ids': ['langchainjs/libs/langchain-textsplitters/src/text_splitter.ts_0_8341', 'langchainjs/docs/core_docs/docs/how_to/character_text_splitter.ipynb_0_4474'], 'nugget_data': [{'nugget_id': '76603417_nugget_0', 'text': 'The `createDocuments` function accepts a second argument, which is an array of objects.', 'relevant_corpus_ids': ['langchainjs/libs/langchain-text

In [14]:
dspy_evaluator_kwargs = {
    "num_threads": 5
}

evaluator(query_writer, **dspy_evaluator_kwargs)

Average Metric: 15.31 / 20 (76.5%): 100%|██████████| 20/20 [00:30<00:00,  1.51s/it]

2025/08/13 21:54:31 INFO dspy.evaluate.evaluate: Average Metric: 15.308333333333334 / 20 (76.5%)


EvaluationResult(score=76.54, results=<list of 20 results>)

In [15]:
import dspy

import logging

# Simple setup for Jupyter
logging.basicConfig(level=logging.INFO, force=True)
logging.getLogger('dspy.teleprompt.gepa').setLevel(logging.INFO)
logging.getLogger('gepa').setLevel(logging.INFO)

# SILENCE the noisy HTTP loggers
logging.getLogger('httpx').setLevel(logging.WARNING)  # Only warnings and errors
logging.getLogger('openai').setLevel(logging.WARNING)
logging.getLogger('weaviate').setLevel(logging.WARNING)
logging.getLogger('httpcore').setLevel(logging.WARNING)

reflection_lm = dspy.LM(
    model="gpt-5",
    temperature=1.0,
    max_tokens=32_000
)

optimizer = dspy.GEPA(
    metric=metric_for_gepa,
    max_metric_calls=500,
    reflection_lm=reflection_lm,
    reflection_minibatch_size=5,
    use_merge=True,
    num_threads=8
)

# there are 30 samples in `trainset` to begin with
trainset=trainset[:15] # these are randomly sampled for Reflective Prompt Mutation
valset=trainset[15:] # these samples create the pareto frontier

optimized_query_expander = optimizer.compile(
    query_writer,
    trainset=trainset,
    valset=valset
)

2025/08/13 21:54:41 INFO dspy.teleprompt.gepa.gepa: Running GEPA for approx 500 metric calls of the program. This amounts to 33.33 full evals on the train+val set.
2025/08/13 21:54:41 INFO dspy.teleprompt.gepa.gepa: Using 15 examples for tracking Pareto scores. You can consider using a sample of the valset to allow GEPA to explore more diverse solutions within the same budget.
2025/08/13 21:54:55 INFO dspy.evaluate.evaluate: Average Metric: 11.833333333333334 / 15 (78.9%)
2025/08/13 21:54:55 INFO dspy.teleprompt.gepa.gepa: Iteration 0: Base program full valset score: 0.788888888888889
2025/08/13 21:54:55 INFO dspy.teleprompt.gepa.gepa: Iteration 1: Selected program 0 score: 0.788888888888889


Average Metric: 4.50 / 5 (90.0%): 100%|██████████| 5/5 [00:07<00:00,  1.48s/it] 

2025/08/13 21:55:03 INFO dspy.evaluate.evaluate: Average Metric: 4.5 / 5 (90.0%)


2025/08/13 21:56:02 INFO dspy.teleprompt.gepa.gepa: Iteration 1: Proposed new text for query_writer: You are given a user’s technical question. Your job is to produce a diverse set of long, highly specific search queries that will help gather the most relevant information from search engines to answer that question.

General requirements
- Output only a flat list (array) of 10–15 search queries (strings). No explanations, no extra text.
- Each query must be long and detailed (aim for 12–25+ words) and include:
  - Exact class/function names, parameters, and error messages (quoted) from the question, when present.
  - Relevant library/framework names and versions if known or commonly implicated.
  - Concrete task wording (what the user is trying to do) and likely solution angles.
  - Multiple phrasings and synonyms to increase recall (e.g., “agent” vs “chain”, “callback” vs “hook”, “retriever tool” vs “vector store tool”).
  - At least some queries with site or intent scoping when appro

Average Metric: 4.25 / 5 (85.0%): 100%|██████████| 5/5 [00:10<00:00,  2.19s/it] 

...

2025/08/13 22:47:14 INFO dspy.evaluate.evaluate: Average Metric: 4.5 / 5 (90.0%)


2025/08/13 22:48:54 INFO dspy.teleprompt.gepa.gepa: Iteration 31: Proposed new text for query_writer: You are given a user’s technical question. Your job is to output a diverse set of long, highly specific search queries that will help gather the most relevant information from search engines to solve the question.

Output format rules
- Output only a flat list (array) of 10–15 search queries (strings). No explanations, no extra text.
- Use a JSON-like Python list of strings.
- Each query must be long and detailed (aim for 12–25+ words).
- Queries must be distinct; avoid near-duplicates and trivial rephrasings.

How to craft the queries
1) Mirror the user’s goal and restate it in several ways across the queries. Reflect concrete tasks they mention and the exact operations they are trying to do, including desired outcomes, constraints, and edge cases.

2) Extract and embed exact class/function names, parameters, method calls, import paths, driver URIs, CLI flags, environment variables, f

Average Metric: 4.75 / 5 (95.0%): 100%|██████████| 5/5 [00:10<00:00,  2.06s/it] 

2025/08/13 22:49:15 INFO dspy.evaluate.evaluate: Average Metric: 4.75 / 5 (95.0%)


2025/08/13 22:50:34 INFO dspy.teleprompt.gepa.gepa: Iteration 32: Proposed new text for query_writer: You are given a user’s technical question. Your job is to output a diverse set of long, highly specific search queries that will help gather the most relevant information from search engines to solve the question.

Output format rules
- Output only a flat list (array) of 10–15 search queries (strings). No explanations, no extra text.
- Use a JSON-like Python list of strings.
- Each query must be long and detailed (aim for 12–25+ words).
- Queries must be distinct; avoid near-duplicates and trivial rephrasings.

How to craft the queries
1) Mirror the user’s goal and restate it in several ways across the queries. Reflect concrete tasks they mention.
2) Extract and embed exact class/function names, parameters, method calls, import paths, driver URIs, CLI flags, and any literal error messages (quoted) from the question. Examples to embed when present:
   - create_sql_agent, AgentExecutor, 

In [16]:
print("GEPA run is finished!")

GEPA run is finished!


In [17]:
optimized_query_expander.save("gepa_optimized_multi_query_writer.json")

In [20]:
evaluator(optimized_query_expander, **dspy_evaluator_kwargs)

Average Metric: 14.81 / 20 (74.0%): 100%|██████████| 20/20 [00:44<00:00,  2.20s/it]

2025/08/13 23:08:07 INFO dspy.evaluate.evaluate: Average Metric: 14.808333333333332 / 20 (74.0%)


EvaluationResult(score=74.04, results=<list of 20 results>)